<a href="https://colab.research.google.com/github/rajavi-mhatre/flyrank-ml/blob/main/work/notebooks/w05_model_ipynb_(final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Logistic Regression is best for classification-based tasks such as this one, where our outcome classes are binary: i) declining ii) not declining. It provides a simple and interpretable model for testing whether pages can be identified on the basis of their decline.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

A random 80/20 train-test split was used, alongside a random state of 42 so that the same split can be reproduced when the notebook runs again.

In [2]:
!git clone https://github.com/rajavi-mhatre/flyrank-ml.git

import pandas as pd

df = pd.read_csv("flyrank-ml/data/raw/content_refresh_anonymized.csv")

print(df.head())
print(df.columns)

print("UNIQUE VALUES: ",df['trend_direction'].unique())

X=['search_volume', 'competition' , 'impressions_90d',
   'clicks_90d', 'pageviews_90d', 'sessions_90d',
   'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d',
   'scroll_events_90d', 'days_with_impressions', 'days_with_sessions',
   'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
   'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
   'content_age_days', 'days_since_last_update',
   'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
   'ai_traffic_pct']

df['declining']= (df['trend_direction'] == 'down').astype(int)
y=['declining']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df[X], df[y], test_size=0.2, random_state=42,stratify=df[y])
X_train=X_train.dropna()
y_train = y_train.loc[X_train.index]
X_test=X_test.dropna()
y_test = y_test.loc[X_test.index]


Cloning into 'flyrank-ml'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 130 (delta 39), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 1.94 MiB | 9.87 MiB/s, done.
Resolving deltas: 100% (39/39), done.
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
model = LogisticRegression()
print(X_train.isna().sum())
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 score:", f1)


search_volume             0
competition               0
impressions_90d           0
clicks_90d                0
pageviews_90d             0
sessions_90d              0
users_90d                 0
engaged_sessions_90d      0
ai_sessions_90d           0
scroll_events_90d         0
days_with_impressions     0
days_with_sessions        0
impressions_last_30d      0
clicks_last_30d           0
sessions_last_30d         0
impressions_prev_30d      0
clicks_prev_30d           0
sessions_prev_30d         0
content_age_days          0
days_since_last_update    0
ctr                       0
avg_position              0
engagement_rate           0
scroll_rate               0
ai_traffic_pct            0
dtype: int64


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Accuracy: 0.9974433893352812
Precision: 0.9996746909564086
Recall: 0.9957874270900843
F1 score: 0.9977272727272727


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
errors = X_test.copy()
errors["actual"] = y_test["declining"]
errors["predicted"] = y_pred
errors = errors[errors["actual"] != errors["predicted"]]
print(errors.head())

coeff = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': model.coef_[0]})
print(coeff.sort_values(by='Coefficient', ascending=False))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)


       search_volume  competition  impressions_90d  clicks_90d  pageviews_90d  \
25967            0.0          0.0              373           0              1   
5503             0.0          0.0                8           0              4   
23348            0.0          0.0               15           0              1   
29439            0.0          0.0               17           0              1   
12910           10.0          1.0               31           0              8   

       sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  \
25967             1          1                     0                1   
5503              4          4                     0                0   
23348             1          1                     0                0   
29439             1          1                     0                0   
12910             8          8                     0                0   

       scroll_events_90d  ...  sessions_prev_30d  content_age_days  \
2596

The model so created assigned positive and negative coefficients to different performance signals. It can be observed from theoutput above that impressions- impressions_prev_30d and impressions_last_30d each have positive and negatve coefficients respectively. It can be inferred therefore, that the model srongly relied on the relationship between previous and recent impressions to determine which pages were declining.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.